# Cold-Start Evaluation
**Hybrid Music Recommendation System — DSCI 441, Spring 2026**

Evaluates four models in the cold-start scenario — each user is known only
by a single song (their most-played), with all other interactions held out.

Models:
- `PopularityBaseline` — ignores seed, returns globally popular songs
- `CFColdStart` — CF has no user vector; degenerates to popularity (benchmarked honestly)
- `Content-only` — k-NN of seed song
- `Hybrid` — cold-start path: `user_id=None`, `seed_song=seed`

Uses the **same 1000 users** as `final_evaluation.ipynb` (loaded from
`results/test_users.pkl`) so warm vs cold comparisons are valid.

**Methodology note:**  
Held-out items are drawn from each user's full MSD listening history, which
extends well beyond the 7,611-song content catalog. This penalises Content and
Hybrid in absolute terms, but the same penalty applies to all models, so
cross-model comparisons remain fair. The key question is relative ranking, not
absolute magnitude.

---
## Section 1 — Setup

In [ ]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'

from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import joblib

from src.data import (
    load_msd_triplets, load_msd_metadata, load_spotify_tracks,
    match_datasets, build_metadata_catalog,
)
from src.models import CFModel, ContentModel, PopularityBaseline, CFColdStart
from src.hybrid import HybridRecommender
from src.evaluation import (
    evaluate_model, bootstrap_ci, paired_bootstrap_test,
    cold_start_test_split,
)

MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

K_VALUES = [5, 10, 20]

In [ ]:
# Load CF model
cf = CFModel.load(MODELS_DIR)
cf_song_to_idx = {sid: idx for idx, sid in cf._idx_to_song.items()}
print(f'CF: {len(cf._user_to_idx):,} users, {len(cf._idx_to_song):,} songs')

In [ ]:
# Load metadata
msd_meta = load_msd_metadata()
spotify  = load_spotify_tracks()
matched  = match_datasets(spotify, msd_meta)
meta_cat = build_metadata_catalog(matched)
print(f'metadata_catalog: {len(meta_cat):,} song_ids')

In [ ]:
# Load Content model
cm = ContentModel.load(MODELS_DIR)
print(f'Content k-NN index: {len(cm._songid_to_idx):,} songs')

In [ ]:
# Fit PopularityBaseline on same filtered triplets
_TRIP_CACHE = RESULTS_DIR / 'triplets_cache.pkl'
if _TRIP_CACHE.exists():
    triplets = joblib.load(_TRIP_CACHE)
    print('Loaded triplets from cache')
else:
    triplets = load_msd_triplets()
    joblib.dump(triplets, _TRIP_CACHE)
    print('Loaded and cached triplets')

pop    = PopularityBaseline().fit(triplets)
cf_cs  = CFColdStart(cf, pop)
hybrid = HybridRecommender(cf, cm, meta_cat, popularity_model=pop, alpha_strategy='adaptive')
print('All models ready')

In [ ]:
# Load the EXACT same 1000 users from final_evaluation.ipynb
# (saved to results/test_users.pkl in that notebook's Section 2)
sampled_users = joblib.load(RESULTS_DIR / 'test_users.pkl')
print(f'Loaded {len(sampled_users)} canonical test users from test_users.pkl')

---
## Section 2 — Cold-Start Test Split

**Design choice:** each user is given exactly one known song as the "seed"
for cold-start inference. The seed is the user's most-played song *from the
Content model's k-NN index* — not necessarily their overall most-played song.

We restrict cold-start eval to users with at least one content-catalog song, so
all four models (Popularity, CFColdStart, Content, Hybrid) can be meaningfully
compared on the same protocol. Users with zero content-catalog interactions are
excluded; their count is reported below.

The held-out set is all other songs the user listened to, excluding the seed.
Held-out items are NOT restricted to the content catalog — this penalty applies
fairly to all models, keeping cross-model comparisons valid.

In [ ]:
_CS_SPLIT_CACHE = RESULTS_DIR / 'cold_start_split.pkl'

if _CS_SPLIT_CACHE.exists():
    cs_data = joblib.load(_CS_SPLIT_CACHE)
    print(f'Loaded cold-start split from cache ({len(cs_data)} users)')
    print(f'  ({len(sampled_users) - len(cs_data)} of the 1000 canonical users excluded '
          f'— no content-catalog seed available)')
else:
    content_ids = set(cm._songid_to_idx.keys())
    sampled_set = set(sampled_users)

    plays = (
        triplets[triplets['uid'].isin(sampled_set)]
        .groupby(['uid', 'sid'])['count']
        .sum()
    )
    users_with_plays = set(plays.index.get_level_values(0))

    cs_data = {}
    excluded_no_content = 0
    for uid in sampled_users:
        if uid not in users_with_plays:
            excluded_no_content += 1
            continue
        # String-index so isin() works cleanly against content_ids (str keys)
        user_plays = plays.loc[uid].sort_values(ascending=False)
        user_plays.index = user_plays.index.astype(str)
        # Restrict seed candidates to content catalog
        content_plays = user_plays[user_plays.index.isin(content_ids)]
        if len(content_plays) == 0:
            excluded_no_content += 1
            continue
        seed_song = content_plays.index[0]          # most-played in content catalog
        held_out  = {s for s in user_plays.index if s != seed_song}
        cs_data[uid] = {
            'seed_song_id':   seed_song,
            'held_out_songs': held_out,
            'history_hidden': True,
        }

    joblib.dump(cs_data, _CS_SPLIT_CACHE)
    print(f'Built and cached cold-start split ({len(cs_data)} users)')
    print(f'Excluded {excluded_no_content} users with zero content-catalog interactions')

# Diagnostics
content_ids   = set(cm._songid_to_idx.keys())
seeds_in_ct   = sum(1 for d in cs_data.values() if d['seed_song_id'] in content_ids)
heldout_in_ct = sum(
    1 for d in cs_data.values()
    if any(s in content_ids for s in d['held_out_songs'])
)
held_sizes = [len(d['held_out_songs']) for d in cs_data.values()]

print(f'\nCold-start split stats:')
print(f'  Total users:                           {len(cs_data)}')
print(f'  Seeds in content catalog:              {seeds_in_ct} '
      f'({100*seeds_in_ct/len(cs_data):.1f}%)  <- should be 100%')
print(f'  Users with any held-out in content:    {heldout_in_ct} '
      f'({100*heldout_in_ct/len(cs_data):.1f}%)  <- upper bound on content hits')
print(f'  Held-out songs per user:  '
      f'median={np.median(held_sizes):.0f}, '
      f'min={min(held_sizes)}, max={max(held_sizes)}')


In [ ]:
# cs_users is a subset of warm_users (filtered to users with content-catalog seeds)
warm_users = set(sampled_users)
cs_users   = set(cs_data.keys())
extra      = cs_users - warm_users
assert not extra, f'{len(extra)} cs_users not in warm_users — bug!'
excluded = warm_users - cs_users
print(f'Cold-start eval: {len(cs_users)} users  '
      f'(excluded {len(excluded)} of 1000 with no content-catalog seed)')


In [ ]:
# Sanity check: genre-conditioned RRF cold-start, 3 users
cs_keys = list(cs_data.keys())
test_users = [cs_keys[0], cs_keys[100], cs_keys[500]]

n_global_fallback = 0
any_both_gte2 = False

for uid in test_users:
    seed = cs_data[uid]['seed_song_id']
    hy_out = hybrid.recommend(user_id=None, seed_song=seed, k=10)

    fallback = hy_out['fallback_used'].iloc[0]
    if fallback == 'global':
        n_global_fallback += 1

    src_counts = hy_out['source'].value_counts().to_dict()
    n_both = src_counts.get('both', 0)
    if n_both >= 2:
        any_both_gte2 = True

    # Seed genre for reporting
    genre_row = meta_cat.loc[meta_cat['song_id'] == seed, 'track_genre']
    genre = genre_row.iloc[0] if not genre_row.empty else 'Unknown'

    print(f'User idx={cs_keys.index(uid):>3}  seed={seed}')
    print(f'  genre={genre!r}  fallback={fallback}')
    print(f'  source breakdown: {src_counts}')
    print(hy_out[['song_id', 'title', 'rrf_score',
                  'content_rank', 'popularity_rank', 'source']].to_string(index=False))
    print()

print(f'Global fallbacks: {n_global_fallback}/3')
print(f'Any user with >= 2 "both" songs: {any_both_gte2}')
